# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io) library. We'll load the dataset via its Croissant schema, review record sets and fields by their `@id`, extract and analyze tabular data, and perform exploratory data analysis including basic visualizations.

**Dataset Source**: 

- Croissant schema URL: [`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure the required library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records using `mlcroissant`. We'll also check the dataset's high-level description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}:\n{metadata.description}\n")

## 2. Data Overview

### Discover available record sets and their `@id`s

Each record set and field has a unique `@id` that we will use for referencing and extraction.

In [ ]:
# List out all record sets and some field info, referencing by `@id`
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    fields = record_set.fields
    print(f"  Number of fields: {len(fields)}")
    for field in fields:
        print(f"    Field: {field.name}, @id: {field.id}, dataType: {getattr(field, 'data_type', 'N/A')}")
    print('-' * 50)

## 3. Data Extraction

We load each record set as a pandas DataFrame, using their `@id` for reproducibility. Select the record set of interest for further analysis.

In [ ]:
# Store DataFrames indexed by record set `@id`
dataframes = {}

for record_set in record_sets:
    rid = record_set.id
    records = list(dataset.records(record_set=rid))  # this yields dicts mapping field @id to value
    if records:
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"Loaded DataFrame for record set '{record_set.name}' (@id: {rid}), shape: {df.shape}")

if not dataframes:
    print("No record sets with tabular data found.")
else:
    # Example: inspect columns of the first available record set
    first_rsid = list(dataframes.keys())[0]
    print(f"\nColumns in record set {first_rsid}:")
    print(dataframes[first_rsid].columns.tolist())
    display(dataframes[first_rsid].head())

## 4. Exploratory Data Analysis (EDA)

This section demonstrates standard EDA using the DataFrame extracted from a record set. We'll filter, normalize, and group data by a chosen field using its `@id`.

In [ ]:
# Select a record set with numeric fields by inspecting loaded DataFrames
if dataframes:
    # Pick the first DataFrame with at least one numeric column for demonstration
    from pandas.api.types import is_numeric_dtype
    numeric_field_id = None
    group_field_id = None
    selected_rsid = None
    
    for rsid, df in dataframes.items():
        for col in df.columns:
            if is_numeric_dtype(df[col]):
                numeric_field_id = col
                selected_rsid = rsid
                # Find a second column of type object (as potential groupby)
                for gc in df.columns:
                    if gc != col and df[gc].dtype == 'object':
                        group_field_id = gc
                        break
                break
        if numeric_field_id:
            break
    
    if numeric_field_id:
        print(f"Selected record set @id: {selected_rsid}")
        print(f"Numeric field @id: {numeric_field_id}")
        if group_field_id:
            print(f"Group field @id: {group_field_id}")

        # Filter records with numeric field > threshold
        threshold = dataframes[selected_rsid][numeric_field_id].mean()  # Use mean as dynamic threshold
        filt = dataframes[selected_rsid][numeric_field_id] > threshold
        filtered_df = dataframes[selected_rsid][filt].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by group_field_id if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found in any loaded record set.")
else:
    print("No data available for EDA.")

## 5. Visualization

Let's plot distributions and relationships of extracted fields. (Requires `matplotlib` and `seaborn`; install if necessary.)

In [ ]:
# Visualization for the example numeric field
if dataframes and numeric_field_id:
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(10, 5))
    sns.histplot(dataframes[selected_rsid][numeric_field_id].dropna(), kde=True, color='skyblue')
    plt.title(f"Distribution of '{numeric_field_id}' in record set '{selected_rsid}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[selected_rsid])
        plt.title(f"Boxplot of '{numeric_field_id}' grouped by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to discover, inspect, and analyze data defined via a Croissant schema. By referencing entities (record sets, fields) by their `@id`, we ensured reproducibility and clarity. Further analysis can explore more in-depth domain questions or be adapted to other Croissant-compliant datasets.